# Semantic Model TOM Catalog

Connect to Fabric semantic models through Semantic Link Labs and TOM, then catalog schema definitions, RLS and object-level security, and direct report dependencies. Model access is read-only and uses the notebook user's Fabric identity; the output snapshots are written to the attached Lakehouse.

Run this notebook interactively. Security collection reads role definitions, not members or user assignments. Confirm the identity can inspect full security metadata using a known protected model. Missing or partial reads remain unknown, not evidence that security is absent.

## Prerequisites

Run this notebook in a Fabric notebook runtime with access to the target semantic model. The runtime identity must have permission to read the model metadata.

In [ ]:
# Install the Semantic Link Labs package if it is not already available.
%pip install -q semantic-link-labs pandas

In [ ]:
import pandas as pd
import sempy.fabric as fabric
from sempy.fabric.exceptions import FabricHTTPException
from sempy_labs import admin
from sempy_labs.tom import connect_semantic_model

## Configuration

Set the scan scope here, then run the notebook top to bottom. Leave both filters as `None` to catalog every workspace and model the notebook identity can see. Outputs are written to the **attached lakehouse** and replace the prior snapshots on each run, so make sure a lakehouse is attached before running.

In [ ]:
# Scan scope: None means every workspace / model visible to the notebook identity.
WORKSPACE_NAME = None  # Optional exact workspace-name filter.
MODEL_NAME = None  # Optional exact model-name filter.
REPORT_WORKSPACE_NAME = None

# Results are written to the lakehouse attached to this notebook.

if WORKSPACE_NAME is not None and not WORKSPACE_NAME.strip():
    raise ValueError("WORKSPACE_NAME must be None or a non-empty workspace name.")
if REPORT_WORKSPACE_NAME is not None and not REPORT_WORKSPACE_NAME.strip():
    raise ValueError("REPORT_WORKSPACE_NAME must be None or a non-empty workspace name.")

## Discover and catalog all workspaces

When `WORKSPACE_NAME` is `None`, the notebook discovers every workspace visible to the Fabric identity, lists its semantic models, and opens a read-only TOM session for each model. Set `WORKSPACE_NAME` or `MODEL_NAME` to narrow the scan.

In [ ]:
def collect_security_definition(model):
    import json

    result = {
        "security_schema_version": 1,
        "scan_status": "failed",
        "error_type": None,
        "role_count": None,
        "rls_filter_count": None,
        "table_ols_count": None,
        "column_ols_count": None,
        "definition_json": None,
    }
    roles = []

    def required_name(value):
        if value is None or not str(value).strip():
            raise ValueError("Missing security object name")
        return str(value)

    def required_enum(value, allowed):
        if value is None or str(value) not in allowed:
            raise ValueError("Unsupported security metadata value")
        return str(value)

    try:
        for role in model.Roles:
            role_definition = {
                "name": required_name(role.Name),
                "model_permission": required_enum(
                    role.ModelPermission, {"None", "Read", "ReadRefresh", "Refresh", "Administrator"}
                ),
                "tables": [],
            }
            for permission in role.TablePermissions:
                expression = permission.FilterExpression
                table_definition = {
                    "table": required_name(permission.Table.Name),
                    "filter_expression": str(expression) if expression is not None else "",
                    "metadata_permission": required_enum(permission.MetadataPermission, {"Default", "None", "Read"}),
                    "columns": [],
                }
                for column_permission in permission.ColumnPermissions:
                    table_definition["columns"].append({
                        "column": required_name(column_permission.Column.Name),
                        "metadata_permission": required_enum(column_permission.MetadataPermission, {"Default", "None", "Read"}),
                    })
                role_definition["tables"].append(table_definition)
            roles.append(role_definition)

        relationships = []
        for relationship in model.Relationships:
            active = relationship.IsActive
            if active is None:
                raise ValueError("Missing relationship active state")
            relationships.append({
                "from_table": required_name(relationship.FromTable.Name),
                "from_column": required_name(relationship.FromColumn.Name),
                "to_table": required_name(relationship.ToTable.Name),
                "to_column": required_name(relationship.ToColumn.Name),
                "is_active": bool(active),
                "cross_filtering_behavior": required_enum(
                    relationship.CrossFilteringBehavior, {"OneDirection", "BothDirections", "Automatic"}
                ),
                "security_filtering_behavior": required_enum(
                    relationship.SecurityFilteringBehavior, {"OneDirection", "BothDirections", "None"}
                ),
                "from_cardinality": required_enum(relationship.FromCardinality, {"One", "Many"}),
                "to_cardinality": required_enum(relationship.ToCardinality, {"One", "Many"}),
            })
        tables = [table for role in roles for table in role["tables"]]
        result.update({
            "scan_status": "complete",
            "role_count": len(roles),
            "rls_filter_count": sum(bool(table["filter_expression"].strip()) for table in tables),
            "table_ols_count": sum(table["metadata_permission"] == "None" for table in tables),
            "column_ols_count": sum(
                column["metadata_permission"] == "None" for table in tables for column in table["columns"]
            ),
            "definition_json": json.dumps({"roles": roles, "relationships": relationships}, sort_keys=True, ensure_ascii=True),
        })
    except Exception as error:
        result["scan_status"] = "partial" if roles else "failed"
        result["error_type"] = type(error).__name__
    return result


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

catalog_scan_id = str(uuid4())
catalog_scanned_at = datetime.now(timezone.utc).isoformat()

try:
    workspaces_df = admin.list_workspaces()
except AttributeError:
    workspaces_df = fabric.list_workspaces()
except FabricHTTPException as error:
    workspaces_df = fabric.list_workspaces()


if workspaces_df.empty:
    raise ValueError("No workspaces are visible to the notebook identity.")

if "Id" in workspaces_df.columns:
    workspace_id_column = "Id"
elif "Workspace Id" in workspaces_df.columns:
    workspace_id_column = "Workspace Id"
else:
    workspace_id_column = "id"

if "Name" in workspaces_df.columns:
    workspace_name_column = "Name"
elif "Workspace Name" in workspaces_df.columns:
    workspace_name_column = "Workspace Name"
else:
    workspace_name_column = "displayName"

if WORKSPACE_NAME is not None:
    workspaces_df = workspaces_df[
        workspaces_df[workspace_name_column].astype(str).str.casefold() == WORKSPACE_NAME.casefold()
    ]

if workspaces_df.empty:
    raise ValueError(f"No matching workspace was found for WORKSPACE_NAME={WORKSPACE_NAME!r}")

datasource_rows = []
table_rows = []
column_rows = []
relationship_rows = []
measure_rows = []
model_rows = []
errors_rows = []
security_rows = []

for _, workspace in workspaces_df.iterrows():
    workspace_id = str(workspace[workspace_id_column])
    workspace_name = str(workspace[workspace_name_column])
    print(f"Scanning workspace: {workspace_name}")

    try:
        datasets_df = fabric.list_datasets(workspace=workspace_id)
    except Exception as error:
        errors_rows.append({
            "workspace_id": workspace_id,
            "workspace_name": workspace_name,
            "model_id": None,
            "model_name": None,
            "error_type": type(error).__name__,
            "error_message": f"Could not list semantic models: {error}",
        })
        continue

    if datasets_df.empty:
        continue

    model_id_column = "Dataset ID" if "Dataset ID" in datasets_df.columns else "Id"
    model_name_column = "Dataset Name" if "Dataset Name" in datasets_df.columns else "Name"

    if MODEL_NAME is not None:
        datasets_df = datasets_df[
            datasets_df[model_name_column].astype(str).str.casefold() == MODEL_NAME.casefold()
        ]

    for _, dataset in datasets_df.iterrows():
        model_id = str(dataset[model_id_column])
        model_name = str(dataset[model_name_column])
        print(f"  Extracting model: {model_name}")
        security_result = {
            "security_schema_version": 1, "scan_status": "failed", "error_type": None,
            "role_count": None, "rls_filter_count": None, "table_ols_count": None,
            "column_ols_count": None, "definition_json": None,
        }
        security_read = False

        try:
            with connect_semantic_model(
                dataset=model_id,
                workspace=workspace_id,
                readonly=True,
            ) as tom:
                model = tom.model
                security_result = collect_security_definition(model)
                security_read = True

                model_rows.append({
                    "workspace_id": workspace_id,
                    "workspace_name": workspace_name,
                    "model_id": model_id,
                    "model_name": model_name,
                    "catalog_scan_id": catalog_scan_id,
                    "compatibility_level": getattr(model, "CompatibilityLevel", None),
                    "default_mode": str(getattr(model, "DefaultMode", None)) if getattr(model, "DefaultMode", None) is not None else None,
                })

                for datasource in model.DataSources:
                    datasource_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "datasource_name": str(getattr(datasource, "Name", "") or ""),
                        "datasource_type": str(getattr(datasource, "Type", None)) if getattr(datasource, "Type", None) is not None else None,
                        "connection_string": getattr(datasource, "ConnectionString", None),
                        "connection_details": str(getattr(datasource, "ConnectionDetails", None)) if getattr(datasource, "ConnectionDetails", None) is not None else None,
                        "impersonation_mode": str(getattr(datasource, "ImpersonationMode", None)) if getattr(datasource, "ImpersonationMode", None) is not None else None,
                        "description": getattr(datasource, "Description", None),
                    })

                # Shared M expressions carry the real source (SQL endpoint / server + database) for
                # Direct Lake and parameterized Power Query models, where DataSources is empty.
                for expression in getattr(model, "Expressions", []):
                    datasource_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "datasource_name": str(getattr(expression, "Name", "") or ""),
                        "datasource_type": "M expression",
                        "connection_string": getattr(expression, "Expression", None),
                        "connection_details": None,
                        "impersonation_mode": None,
                        "description": getattr(expression, "Description", None),
                    })

                # Inline Power Query partition sources carry the connection for import models.
                # Entity / Direct Lake partitions only reference a shared expression (captured above),
                # so they are skipped here to avoid matching different sources on a generic name.
                for table in model.Tables:
                    for partition in getattr(table, "Partitions", []):
                        partition_source = getattr(partition, "Source", None)
                        m_expression = getattr(partition_source, "Expression", None) if partition_source is not None else None
                        if not m_expression:
                            continue
                        datasource_rows.append({
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "model_id": model_id,
                            "model_name": model_name,
                            "datasource_name": str(getattr(partition, "Name", "") or ""),
                            "datasource_type": type(partition_source).__name__,
                            "connection_string": str(m_expression),
                            "connection_details": None,
                            "impersonation_mode": None,
                            "description": None,
                        })

                for table in model.Tables:
                    table_name = str(getattr(table, "Name", "") or "")
                    table_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "table_name": table_name,
                        "description": getattr(table, "Description", None),
                        "is_hidden": getattr(table, "IsHidden", None),
                    })

                    for column in getattr(table, "Columns", []):
                        column_rows.append({
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "model_id": model_id,
                            "model_name": model_name,
                            "table_name": table_name,
                            "column_name": str(getattr(column, "Name", "") or ""),
                            "data_type": str(getattr(column, "DataType", None)) if getattr(column, "DataType", None) is not None else None,
                            "description": getattr(column, "Description", None),
                            "is_hidden": getattr(column, "IsHidden", None),
                        })

                    for measure in getattr(table, "Measures", []):
                        measure_rows.append({
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "model_id": model_id,
                            "model_name": model_name,
                            "table_name": table_name,
                            "measure_name": str(getattr(measure, "Name", "") or ""),
                            "expression": getattr(measure, "Expression", None),
                            "format_string": getattr(measure, "FormatString", None),
                            "description": getattr(measure, "Description", None),
                            "is_hidden": getattr(measure, "IsHidden", None),
                        })

                for relationship in getattr(model, "Relationships", []):
                    from_table = getattr(relationship, "FromTable", None)
                    from_column = getattr(relationship, "FromColumn", None)
                    to_table = getattr(relationship, "ToTable", None)
                    to_column = getattr(relationship, "ToColumn", None)
                    relationship_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "relationship_name": str(getattr(relationship, "Name", "") or ""),
                        "from_table": str(getattr(from_table, "Name", "") or ""),
                        "from_column": str(getattr(from_column, "Name", "") or ""),
                        "to_table": str(getattr(to_table, "Name", "") or ""),
                        "to_column": str(getattr(to_column, "Name", "") or ""),
                        "cross_filtering_behavior": str(getattr(relationship, "CrossFilteringBehavior", None)) if getattr(relationship, "CrossFilteringBehavior", None) is not None else None,
                        "is_active": getattr(relationship, "IsActive", None),
                    })
        except Exception as error:
            if not security_read:
                security_result["error_type"] = type(error).__name__
            errors_rows.append({
                "workspace_id": workspace_id,
                "workspace_name": workspace_name,
                "model_id": model_id,
                "model_name": model_name,
                "error_type": type(error).__name__,
                "error_message": str(error),
            })
        finally:
            security_rows.append({
                "workspace_id": workspace_id, "workspace_name": workspace_name,
                "model_id": model_id, "model_name": model_name,
                "catalog_scan_id": catalog_scan_id, "scanned_at": catalog_scanned_at,
                **security_result,
            })

models_df = pd.DataFrame(model_rows)
datasources_df = pd.DataFrame(datasource_rows)
tables_df = pd.DataFrame(table_rows)
columns_df = pd.DataFrame(column_rows)
relationships_df = pd.DataFrame(relationship_rows)
measures_df = pd.DataFrame(measure_rows)
errors_df = pd.DataFrame(errors_rows)
security_df = pd.DataFrame(security_rows)

print(f"Cataloged {len(models_df)} models across {len(workspaces_df)} workspaces.")
print(f"Datasources: {len(datasources_df)} | Tables: {len(tables_df)} | Columns: {len(columns_df)} | Relationships: {len(relationships_df)} | Measures: {len(measures_df)}")
print(f"Security definitions complete: {sum(row['scan_status'] == 'complete' for row in security_rows)}/{len(security_rows)} model scans.")
if not errors_df.empty:
    print(f"Workspace/model errors: {len(errors_df)}")


In [ ]:
def collect_report_dependencies(client, workspaces, models, scan_id, scanned_at):
    from urllib.parse import quote, urljoin, urlsplit

    model_index = {str(model["model_id"]).strip().casefold(): model for model in models}
    dependencies = []
    scans = []
    for workspace in workspaces:
        workspace_id = str(workspace["workspace_id"]).strip().casefold()
        workspace_name = str(workspace["workspace_name"])
        endpoint = "/v1.0/myorg/groups/" + quote(workspace_id, safe="") + "/reports"
        next_url = "https://api.powerbi.com" + endpoint
        visited = set()
        workspace_reports = {}
        scan_status = "complete"
        error_type = None
        try:
            while next_url:
                parsed = urlsplit(next_url)
                if (parsed.scheme != "https" or parsed.netloc != "api.powerbi.com"
                        or parsed.path != endpoint or parsed.fragment or next_url in visited):
                    raise ValueError("Invalid report continuation")
                visited.add(next_url)
                response = client.get(parsed.path + ("?" + parsed.query if parsed.query else ""))
                response.raise_for_status()
                payload = response.json()
                if not isinstance(payload, dict) or not isinstance(payload.get("value"), list):
                    raise ValueError("Invalid report collection")
                for report in payload["value"]:
                    if not isinstance(report, dict) or not isinstance(report.get("id"), str) or not report["id"].strip():
                        raise ValueError("Missing report identity")
                    report_id = report["id"].strip().casefold()
                    raw_model_id = report.get("datasetId")
                    if raw_model_id is not None and not isinstance(raw_model_id, str):
                        raise ValueError("Invalid model identity")
                    model_id = raw_model_id.strip().casefold() if raw_model_id else None
                    report_type = report.get("reportType") or "PowerBIReport"
                    target = model_index.get(model_id)
                    if report_type != "PowerBIReport":
                        binding_status = "unsupported_report_type"
                    elif not model_id:
                        binding_status = "missing_dataset_id"
                    elif target is None:
                        binding_status = "uncataloged_model"
                    else:
                        binding_status = "cataloged_model"
                    dependency = {
                        "scan_id": scan_id,
                        "scanned_at": scanned_at,
                        "report_workspace_id": workspace_id,
                        "report_workspace_name": workspace_name,
                        "report_id": report_id,
                        "report_name": str(report.get("name") or report_id),
                        "report_type": str(report_type),
                        "report_url": "https://app.powerbi.com/groups/" + quote(workspace_id, safe="") + "/reports/" + quote(report_id, safe=""),
                        "model_id": model_id,
                        "model_workspace_id": str(target["workspace_id"]) if target else None,
                        "model_workspace_name": str(target["workspace_name"]) if target else None,
                        "model_name": str(target["model_name"]) if target else None,
                        "binding_status": binding_status,
                        "is_cross_workspace": workspace_id != str(target["workspace_id"]).casefold() if target else None,
                    }
                    if report_id in workspace_reports and workspace_reports[report_id]["model_id"] != model_id:
                        raise ValueError("Report binding changed during scan")
                    workspace_reports[report_id] = dependency
                continuation = payload.get("@odata.nextLink")
                if continuation is not None and not isinstance(continuation, str):
                    raise ValueError("Invalid report continuation")
                next_url = urljoin("https://api.powerbi.com" + endpoint, continuation) if continuation else None
        except Exception as error:
            scan_status = "partial" if workspace_reports else "failed"
            error_type = type(error).__name__
        records = list(workspace_reports.values())
        unresolved_count = sum(record["binding_status"] == "missing_dataset_id" for record in records)
        unsupported_count = sum(record["binding_status"] == "unsupported_report_type" for record in records)
        if scan_status == "complete" and (unresolved_count or unsupported_count):
            scan_status = "partial"
        dependencies.extend(records)
        scans.append({
            "scan_id": scan_id,
            "scanned_at": scanned_at,
            "report_workspace_id": workspace_id,
            "report_workspace_name": workspace_name,
            "scan_status": scan_status,
            "report_count": len(records),
            "bound_report_count": sum(record["binding_status"] in ("cataloged_model", "uncataloged_model") for record in records),
            "unresolved_report_count": unresolved_count,
            "unsupported_report_count": unsupported_count,
            "error_type": error_type,
        })
    return dependencies, scans

In [ ]:
def scan_report_catalog(fabric, models, workspace_name=None):
    from datetime import datetime, timezone
    from uuid import uuid4

    scan_id = str(uuid4())
    scanned_at = datetime.now(timezone.utc).isoformat()
    scope = "workspace_filter:" + workspace_name if workspace_name is not None else "visible_workspaces"
    try:
        workspace_records = fabric.list_workspaces().to_dict("records")
        workspaces = []
        for record in workspace_records:
            workspace_id = next((record[key] for key in ("Id", "Workspace Id", "id") if key in record), None)
            name = next((record[key] for key in ("Name", "Workspace Name", "displayName") if key in record), None)
            if not workspace_id or not isinstance(name, str):
                raise ValueError("Missing report workspace identity")
            if workspace_name is None or name.casefold() == workspace_name.casefold():
                workspaces.append({"workspace_id": str(workspace_id), "workspace_name": name})
        if not workspaces:
            raise ValueError("No report workspaces in scope")
        reports, scans = collect_report_dependencies(
            fabric.PowerBIRestClient(), workspaces, models, scan_id, scanned_at
        )
    except Exception as error:
        reports = []
        scans = [{
            "scan_id": scan_id,
            "scanned_at": scanned_at,
            "report_workspace_id": None,
            "report_workspace_name": None,
            "scan_status": "failed",
            "report_count": 0,
            "bound_report_count": 0,
            "unresolved_report_count": 0,
            "unsupported_report_count": 0,
            "error_type": type(error).__name__,
        }]
    for scan in scans:
        scan["scan_scope"] = scope
    return reports, scans

report_rows, report_scan_rows = scan_report_catalog(
    fabric, models_df.to_dict("records"), REPORT_WORKSPACE_NAME
 )
reports_df = pd.DataFrame(report_rows)
report_scans_df = pd.DataFrame(report_scan_rows)
print(f"Report snapshot: {len(reports_df)} reports; {sum(scan['scan_status'] == 'complete' for scan in report_scan_rows)}/{len(report_scan_rows)} report workspace scans complete.")

## Review and persist the catalog

The catalog is workspace-wide, and each DataFrame includes model context. `errors_df` records models that could not be read so the run still completes for the rest. The next cell previews the results (starting with a per-workspace model count), and the final cell writes them as Delta tables to the attached lakehouse.

In [ ]:
# Per-workspace model counts give a quick overview before the detailed frames.
if not models_df.empty:
    workspace_summary = (
        models_df.groupby("workspace_name")
        .size()
        .reset_index(name="model_count")
        .sort_values("model_count", ascending=False)
        .reset_index(drop=True)
    )
    display(workspace_summary)

display(models_df)
display(datasources_df)
display(tables_df)
display(columns_df)
display(relationships_df)
display(measures_df)
display(errors_df)

In [ ]:
catalog_outputs = {
    "semantic_models": models_df,
    "semantic_model_datasources": datasources_df,
    "semantic_model_tables": tables_df,
    "semantic_model_columns": columns_df,
    "semantic_model_relationships": relationships_df,
    "semantic_model_measures": measures_df,
    "semantic_model_catalog_errors": errors_df,
    "semantic_model_report_dependencies": reports_df,
    "semantic_model_report_scan": report_scans_df,
    "semantic_model_security": security_df,
}

identity_schema = "workspace_id STRING, workspace_name STRING, model_id STRING, model_name STRING"
catalog_schemas = {
    "semantic_models": identity_schema + ", compatibility_level BIGINT, default_mode STRING, catalog_scan_id STRING",
    "semantic_model_datasources": identity_schema + ", datasource_name STRING, datasource_type STRING, connection_string STRING, connection_details STRING, impersonation_mode STRING, description STRING",
    "semantic_model_tables": identity_schema + ", table_name STRING, description STRING, is_hidden BOOLEAN",
    "semantic_model_columns": identity_schema + ", table_name STRING, column_name STRING, data_type STRING, description STRING, is_hidden BOOLEAN",
    "semantic_model_relationships": identity_schema + ", relationship_name STRING, from_table STRING, from_column STRING, to_table STRING, to_column STRING, cross_filtering_behavior STRING, is_active BOOLEAN",
    "semantic_model_measures": identity_schema + ", table_name STRING, measure_name STRING, expression STRING, format_string STRING, description STRING, is_hidden BOOLEAN",
    "semantic_model_catalog_errors": identity_schema + ", error_type STRING, error_message STRING",
    "semantic_model_report_dependencies": "scan_id STRING, scanned_at STRING, report_workspace_id STRING, report_workspace_name STRING, report_id STRING, report_name STRING, report_type STRING, report_url STRING, model_id STRING, model_workspace_id STRING, model_workspace_name STRING, model_name STRING, binding_status STRING, is_cross_workspace BOOLEAN",
    "semantic_model_report_scan": "scan_id STRING, scanned_at STRING, report_workspace_id STRING, report_workspace_name STRING, scan_status STRING, report_count BIGINT, bound_report_count BIGINT, unresolved_report_count BIGINT, unsupported_report_count BIGINT, error_type STRING, scan_scope STRING",
    "semantic_model_security": identity_schema + ", catalog_scan_id STRING, scanned_at STRING, security_schema_version BIGINT, scan_status STRING, error_type STRING, role_count BIGINT, rls_filter_count BIGINT, table_ols_count BIGINT, column_ols_count BIGINT, definition_json STRING",
}

for table_name, frame in catalog_outputs.items():
    records = frame.to_dict("records")
    for record in records:
        for field, value in record.items():
            if pd.isna(value):
                record[field] = None
        for field in ("compatibility_level", "security_schema_version", "role_count", "rls_filter_count", "table_ols_count", "column_ols_count"):
            if record.get(field) is not None:
                record[field] = int(record[field])
    spark.createDataFrame(records, schema=catalog_schemas[table_name]).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")


## Next steps

Check the security scan completion count and `semantic_model_security.scan_status`. A complete scan with zero roles is different from a failed or partial read. Resolve metadata access before interpreting protected models as role-free.

Open **002_semantic_model_similarity.ipynb**, attached to this same Lakehouse, and run it to calculate schema, security, and combined similarity. Then run **003_semantic_model_similarity_results.ipynb** to review the scores, security differences, and schema coverage. Security definitions are sensitive metadata; keep the Lakehouse and results within the intended access scope.